In [1]:
import os, warnings,sys,time    

CUDA_VISIBLE_DEVICES=""
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0,'../../uqmodels/abench')
sys.path.insert(0,'../../n5_uqmodels/')
import abench
import uqmodels
import yaml

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Specification of Data :
import yaml
config_benchmark_path = "config/config_benchmark.yaml"
with open(config_benchmark_path) as f:
    config_benchmark = yaml.safe_load(f)
    
for key,value in config_benchmark.items():
    print(key,value)

# Metric Specification 
from src.metric import base_rmse,ABMetricGeneric
#['Id'0, 'timestamp'1, 'positionX'2, 'positionY'3, 'positionZ'4, 'sizeX'5,'sizeY+', 'sizeZ7', 'VelX'8, 'VelY'9, 'VelZ'10, 'Vel'11, 'Class'12, 'rot_x'13,'rot_y'14, 'edge'15, 'ts'16, 'departure'17, 'destination18, 'dataset'19, 'set'20,'filename'21, 'seqId'22, 'length'23, 'cat_length'24, 'cat_edge'25],
dict_sets_configs_grid1={'context_mask':[0],'context_dim_mask':1,'context_variable_ids':[[24]]}
metrics = [ABMetricGeneric(base_rmse, name="RMSE",reduce=True),
           ABMetricGeneric(base_rmse, name="RMSE_grid",dict_sets_config=dict_sets_configs_grid1,reduce=True)]

datasets ['new_dataset10']
validation_config {'Random_pos_offset_low': {'constraint_selection': [['dataset', ['new_dataset10_random_pos_offset_low']]], 'constraint_rejection': []}, 'Random_pos_offset_median': {'constraint_selection': [['dataset', ['new_dataset10_random_pos_offset_median']]], 'constraint_rejection': []}, 'Random_pos_offset_high': {'constraint_selection': [['dataset', ['new_dataset10_random_pos_offset_high']]], 'constraint_rejection': []}}
sets_norm_scale ['r2_set_0', 'r2_set_1', 'r2_set_2', 'r2_set_3', 'r2_set_4']
sets_cv_exp ['r2_set_0', 'r2_set_1', 'r2_set_2', 'r2_set_3', 'r2_set_4']
Components_config {'DenseAE': 'config/config_ae.yaml', 'DenseVAE': 'config/config_vae.yaml', 'ConvAE': 'config/config_ae.yaml', 'ConvVAE': 'config/config_vae.yaml', 'TimeAE': 'config/config_ae.yaml', 'TimeVAE': 'config/config_vae.yaml'}


2026-02-17 16:17:02.272224: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]


2026-02-17 13:20:17.213118: E tensorflow/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:
# Spécification of DataExperiment Plan
from src.data_loader import get_DataExperiment
DataExperiment = get_DataExperiment(config_benchmark)

# Spécification of Models Candidates
from src.component import build_params,get_model_constructor

dict_comp = {}
for name,config_path in config_benchmark['Components_config'].items():
    dict_comp[name]={'module':get_model_constructor(name),'parameters':build_params(config_path)}

# Specification of the Component candidate list :
exp_design=[]
for key,model_builder in dict_comp.items():
    subexp_design=[{'name':key,'model':key}]
    exp_design.append(subexp_design)

from src.component import ComponentAE
dict_exp={'Component': ComponentAE,
          'tuning_scheme' : {},
          'model': dict_comp,
          'exp_design':exp_design}

100%|██████████| 25/25 [00:07<00:00,  3.17it/s]


r2_set_0
r2_set_1
r2_set_2
r2_set_3
r2_set_4
Train data_Train_LOSO_r2_set_0
Test data_Test_LOSO_r2_set_0
Test data_Valid_LOSO_r2_set_0_Random_pos_offset_low
Test data_Valid_LOSO_r2_set_0_Random_pos_offset_median
Test data_Valid_LOSO_r2_set_0_Random_pos_offset_high
Train data_Train_LOSO_r2_set_1
Test data_Test_LOSO_r2_set_1
Test data_Valid_LOSO_r2_set_1_Random_pos_offset_low
Test data_Valid_LOSO_r2_set_1_Random_pos_offset_median
Test data_Valid_LOSO_r2_set_1_Random_pos_offset_high
Train data_Train_LOSO_r2_set_2
Test data_Test_LOSO_r2_set_2
Test data_Valid_LOSO_r2_set_2_Random_pos_offset_low
Test data_Valid_LOSO_r2_set_2_Random_pos_offset_median
Test data_Valid_LOSO_r2_set_2_Random_pos_offset_high
Train data_Train_LOSO_r2_set_3
Test data_Test_LOSO_r2_set_3
Test data_Valid_LOSO_r2_set_3_Random_pos_offset_low
Test data_Valid_LOSO_r2_set_3_Random_pos_offset_median
Test data_Valid_LOSO_r2_set_3_Random_pos_offset_high
Train data_Train_LOSO_r2_set_4
Test data_Test_LOSO_r2_set_4
Test data_Valid

# Run benchmark with train step

In [5]:
from src.metric import base_rmse,ABMetricGeneric
AB_RMSE = ABMetricGeneric(metric=base_rmse,name="rmse", mask=None, dim_mask=None, list_ctx_constraint=None,reduce=True)
list_metrics = [AB_RMSE]

In [6]:
# Metric Specification 
from abench.benchmark.benchmark import benchmark
storing = 'Results'
benchmark(storing=storing,
          ABDataExperiment=DataExperiment,
          dict_exp=dict_exp,
          # Component_class,
          list_metrics=list_metrics,verbose=True)

n_component : [{'name': 'DenseAE', 'model': 'DenseAE'}]
Train ondata_Train_LOSO_r2_set_0


2026-02-17 16:18:04.052450: E tensorflow/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


<function build_MSE_loss.<locals>.MSE_loss at 0x7ee9a380c9d0>


100%|██████████| 48/48 [00:10<00:00,  4.61it/s]


Epoch 1/10
1028/1028 [==============================] - 7s 6ms/step - loss: 0.0584 - reconstruction_loss: 0.0584 - lr: 0.0010
Epoch 2/10
1028/1028 [==============================] - 6s 5ms/step - loss: 0.0214 - reconstruction_loss: 0.0214 - lr: 0.0010
Epoch 3/10
1028/1028 [==============================] - 6s 6ms/step - loss: 0.0167 - reconstruction_loss: 0.0167 - lr: 0.0010
Epoch 4/10
1028/1028 [==============================] - 6s 6ms/step - loss: 0.0143 - reconstruction_loss: 0.0143 - lr: 0.0010
Epoch 5/10
1028/1028 [==============================] - 6s 6ms/step - loss: 0.0129 - reconstruction_loss: 0.0129 - lr: 0.0010
Epoch 6/10
1028/1028 [==============================] - 5s 5ms/step - loss: 0.0118 - reconstruction_loss: 0.0118 - lr: 0.0010
Epoch 7/10
1028/1028 [==============================] - 6s 6ms/step - loss: 0.0111 - reconstruction_loss: 0.0111 - lr: 0.0010
Epoch 8/10
1028/1028 [==============================] - 5s 5ms/step - loss: 0.0104 - reconstruction_loss: 0.0104 - lr:

...

No file found for Results/data_Train_LOSO_r2_set_3/DenseAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_3/DenseVAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_3/ConvAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_3/ConvVAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_3/TimeAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_3/TimeVAE/dictperf (tried .joblib/.p/.parquet/.csv)


100%|██████████| 48/48 [00:14<00:00,  3.36it/s]


No file found for Results/data_Train_LOSO_r2_set_4/DenseAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_4/DenseVAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_4/ConvAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_4/ConvVAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_4/TimeAE/dictperf (tried .joblib/.p/.parquet/.csv)
No file found for Results/data_Train_LOSO_r2_set_4/TimeVAE/dictperf (tried .joblib/.p/.parquet/.csv)


100%|██████████| 5/5 [00:00<00:00,  7.95it/s]


DenseAE rmse Train nan ± nan | TEST 0.119 ± 0.033
DenseVAE rmse Train nan ± nan | TEST 0.317 ± 0.082
ConvAE rmse Train nan ± nan | TEST 0.119 ± 0.038
ConvVAE rmse Train nan ± nan | TEST 0.324 ± 0.088
TimeAE rmse Train nan ± nan | TEST 0.121 ± 0.046
TimeVAE rmse Train nan ± nan | TEST 0.325 ± 0.088


# Run only inference for a new validation set

In [8]:
# Metric Specification 
from src.metric import base_rmse,ABMetricGeneric
AB_RMSE = ABMetricGeneric(metric=base_rmse,name="rmse", mask=None, dim_mask=None, list_ctx_constraint=None,reduce=True)
list_metrics = [AB_RMSE]
from abench.benchmark import benchmark

storing = 'Results/'
from src.data_loader import get_DataExperiment
config_benchmark['validation_config'] = {'Noisy_eps': {'constraint_selection': [['dataset',['dataset1_noisy_eps']]], 'constraint_rejection': []}}
DataExperiment = get_DataExperiment(config_benchmark,with_test=False)
DataExperiment.name = 'cv_experiment_additional'

# List of component name
list_component_name = list(config_benchmark['Components_config'].keys())

# Dict making association between name and class
from src.component import ComponentAE
Component_class_dict = {}
for name_model in list_component_name:
    Component_class_dict[name_model] = ComponentAE

benchmark.inference(storing=storing,
                    ABDataExperiment=DataExperiment,
                    list_component_name = list_component_name,
                    Component_class_dict = Component_class_dict,
                    list_metrics=list_metrics,verbose=True)

100%|██████████| 5/5 [00:03<00:00,  1.34it/s]


set_0
set_1
set_2
set_3
set_4
Train data_Train_LOSO_set_0
Test data_Valid_LOSO_set_0_Noisy_eps
Train data_Train_LOSO_set_1
Test data_Valid_LOSO_set_1_Noisy_eps
Train data_Train_LOSO_set_2
Test data_Valid_LOSO_set_2_Noisy_eps
Train data_Train_LOSO_set_3
Test data_Valid_LOSO_set_3_Noisy_eps
Train data_Train_LOSO_set_4
Test data_Valid_LOSO_set_4_Noisy_eps
{'class_name': 'DenseAE', 'init_kwargs': {'basic': True, 'hidden': [128, 128], 'input_dim': 4, 'latent_dim': 12, 'seq_len': 50, 'training_params': {'batch_size': 128, 'epochs': 2, 'optimizer': 'nadam'}}, 'name': 'dense_ae'}
<function build_MSE_loss.<locals>.MSE_loss at 0x788946d00f70>
<function build_MSE_loss.<locals>.MSE_loss at 0x788946d00af0>
Test ondata_Valid_LOSO_set_0_Noisy_eps


100%|██████████| 1/1 [00:00<00:00,  1.54it/s]


{'class_name': 'DenseAE', 'init_kwargs': {'basic': True, 'hidden': [128, 128], 'input_dim': 4, 'latent_dim': 12, 'seq_len': 50, 'training_params': {'batch_size': 128, 'epochs': 2, 'optimizer': 'nadam'}}, 'name': 'dense_ae'}
<function build_MSE_loss.<locals>.MSE_loss at 0x788946d01120>
<function build_MSE_loss.<locals>.MSE_loss at 0x788946ee3010>
Test ondata_Valid_LOSO_set_1_Noisy_eps


100%|██████████| 1/1 [00:00<00:00,  1.61it/s]


...

{'class_name': 'TimeVAE', 'init_kwargs': {'conv_filters': [50, 100, 200], 'custom_seas': None, 'input_dim': 4, 'latent_dim': 12, 'seq_len': 50, 'training_params': {'batch_size': 128, 'epochs': 2, 'optimizer': 'nadam'}, 'trend_poly': 0, 'use_residual_conn': True}, 'name': 'time_vae'}
<function build_MSE_loss.<locals>.MSE_loss at 0x7889a967fac0>
<function build_MSE_loss.<locals>.MSE_loss at 0x788946e801f0>
Test ondata_Valid_LOSO_set_3_Noisy_eps


100%|██████████| 1/1 [00:00<00:00,  1.41it/s]


{'class_name': 'TimeVAE', 'init_kwargs': {'conv_filters': [50, 100, 200], 'custom_seas': None, 'input_dim': 4, 'latent_dim': 12, 'seq_len': 50, 'training_params': {'batch_size': 128, 'epochs': 2, 'optimizer': 'nadam'}, 'trend_poly': 0, 'use_residual_conn': True}, 'name': 'time_vae'}
<function build_MSE_loss.<locals>.MSE_loss at 0x7889a9663130>
<function build_MSE_loss.<locals>.MSE_loss at 0x788946e81360>
Test ondata_Valid_LOSO_set_4_Noisy_eps


100%|██████████| 1/1 [00:00<00:00,  1.22it/s]


DenseAE rmse Train nan ± nan | TEST 0.081 ± 0.006
DenseVAE rmse Train nan ± nan | TEST 0.222 ± 0.005
ConvAE rmse Train nan ± nan | TEST 0.089 ± 0.009
ConvVAE rmse Train nan ± nan | TEST 0.22 ± 0.003
TimeAE rmse Train nan ± nan | TEST 0.085 ± 0.014
TimeVAE rmse Train nan ± nan | TEST 0.218 ± 0.011


In [9]:
# Update full data experiment 
import yaml
config_benchmark_path = "config/config_benchmark.yaml"
with open(config_benchmark_path) as f:
    config_benchmark = yaml.safe_load(f)

storing = 'Results/'
from src.data_loader import get_DataExperiment
config_benchmark['validation_config']['Noisy_eps'] = {'constraint_selection': [['dataset',['dataset1_noisy_eps']]], 'constraint_rejection': []}
DataExperiment = get_DataExperiment(config_benchmark,with_test=True)
from abench.store.api import store_ABDataExperiment
store_ABDataExperiment(storing,DataExperiment)

100%|██████████| 5/5 [00:03<00:00,  1.41it/s]


set_0
set_1
set_2
set_3
set_4
Train data_Train_LOSO_set_0
Test data_Test_LOSO_set_0
Test data_Valid_LOSO_set_0_Noisy
Test data_Valid_LOSO_set_0_Piecewise
Test data_Valid_LOSO_set_0_Piecewise_on_cat2
Test data_Valid_LOSO_set_0_Noisy_eps
Train data_Train_LOSO_set_1
Test data_Test_LOSO_set_1
Test data_Valid_LOSO_set_1_Noisy
Test data_Valid_LOSO_set_1_Piecewise
Test data_Valid_LOSO_set_1_Piecewise_on_cat2
Test data_Valid_LOSO_set_1_Noisy_eps
Train data_Train_LOSO_set_2
Test data_Test_LOSO_set_2
Test data_Valid_LOSO_set_2_Noisy
Test data_Valid_LOSO_set_2_Piecewise
Test data_Valid_LOSO_set_2_Piecewise_on_cat2
Test data_Valid_LOSO_set_2_Noisy_eps
Train data_Train_LOSO_set_3
Test data_Test_LOSO_set_3
Test data_Valid_LOSO_set_3_Noisy
Test data_Valid_LOSO_set_3_Piecewise
Test data_Valid_LOSO_set_3_Piecewise_on_cat2
Test data_Valid_LOSO_set_3_Noisy_eps
Train data_Train_LOSO_set_4
Test data_Test_LOSO_set_4
Test data_Valid_LOSO_set_4_Noisy
Test data_Valid_LOSO_set_4_Piecewise
Test data_Valid_LOSO_s